# 04 — UTXO Snapshots
## Snapshots agrégés de l'UTXO set (HODL waves, distribution par âge et valeur)

**Prérequis** : Bitcoin Core synchronisé + `bitcoin-utxo-dump` installé
**Durée** : ~20-30 min par snapshot
**Mode agrégé** : stocke uniquement les distributions (pas les UTXOs individuels)

In [ ]:
import os, sys, time
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.collectors.bitcoin_core import (
    get_rpc, get_chain_info,
    extract_utxo_snapshot_aggregated, run_utxo_snapshots,
)

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()
config = Config()

In [ ]:
# Vérification Bitcoin Core
rpc = get_rpc()
info = get_chain_info(rpc)
print(f"✅ Chain tip: {info['blocks']:,}")
print(f"   Synced: {'✅' if info['blocks']==info['headers'] else '⏳'}")

# Vérifier bitcoin-utxo-dump
import subprocess
try:
    r = subprocess.run(["bitcoin-utxo-dump", "--help"], capture_output=True, timeout=5)
    print("✅ bitcoin-utxo-dump disponible")
except FileNotFoundError:
    print("❌ bitcoin-utxo-dump non installé")
    print("   wget https://github.com/in3rsha/bitcoin-utxo-dump/releases/download/v0.1.2/bitcoin-utxo-dump-linux-amd64")
    print("   chmod +x bitcoin-utxo-dump-linux-amd64 && mv bitcoin-utxo-dump-linux-amd64 /usr/local/bin/bitcoin-utxo-dump")

## Snapshot UTXO actuel (mode agrégé)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Snapshot UTXO au bloc actuel
# ════════════════════════════════════════════════════════════════════════
print(f"▶ Lancement du snapshot UTXO à la hauteur {info['blocks']:,}")
print(f"  Durée estimée : 20-30 min")
print()

t_start = time.time()
result = extract_utxo_snapshot_aggregated(rpc, storage, info["blocks"], config)
duration = time.time() - t_start

if result.get("status") == "success":
    print(f"\n✅ Snapshot UTXO terminé en {duration/60:.1f} min")
    print(f"   UTXOs totaux : {result['utxo_count']:,}")
    print(f"   BTC total    : {result['total_btc']:.2f}")
    print(f"   Taille       : {result['size_bytes']/1e6:.1f} MB")
else:
    print(f"❌ Erreur : {result}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Vérification du snapshot
# ════════════════════════════════════════════════════════════════════════
files = storage.list_files("raw/blockchain/utxo_snapshots/")
print(f"Snapshots UTXO sur GCS : {len(files)}")
for f in files[-5:]:
    print(f"  {f}")

if files:
    df = storage.download_parquet(files[-1])
    print(f"\nDernier snapshot :")
    print(df.T.to_string())